# OpenRCA — risultati degli agenti e fattibilità R03/R04/R05
**Presentazione per il relatore • 3 novembre 2026 • demo di sola lettura**

Questo notebook raccoglie due **esperimenti congelati** con `gemini-3.5-flash-lite` (8 casi appaiati ciascuno, **coorti distinte**) e gli audit esplorativi GT-free A/1–A/6 su R03, R04 e R05.

**Nessuna chiamata Gemini; nessun nuovo scoring; nessuna lettura diretta di `record.csv` o `query.csv`; nessuna modifica dei risultati su Drive.** I grafici degli agenti sono ricalcolati **solo** da scoring già congelati, con controlli SHA-256. I grafici di rete rappresentano i conteggi degli audit A/4/A/6 del 3 novembre; quando sono disponibili i JSON integrali vengono ricontrollati.

**Attenzione:** non mettere sullo stesso asse gli score di esperimenti condotti su coorti diverse.

## 1. Le architetture, in ordine di sviluppo

| Agente | Meccanismo principale | Valutazione quantitativa documentata |
|---|---|---|
| **A0** | Agente esplorativo con cinque tool | Nessun confronto appaiato congelato in questa demo |
| **A1_V2** | Evidence engineering senza otto regole | Nessun confronto appaiato congelato in questa demo |
| **A1-R8** | Evidence engineering + R01, R02, R06–R11 | Esperimento 1; nello scoring si chiama `A1` |
| **A1_VERIFY** | A1-R8 + planner, massimo 4 round LLM esplorativi + 2 verifiche | Esperimenti 1 e 2, su **coorti differenti** |
| **A1_VERIFY_ST** | A1_VERIFY + contesto strutturale-temporale solo nella finalizzazione | Esperimento 2 |

Cinque **tool** di consultazione ≠ otto **regole** deterministiche. Le regole forniscono evidenza osservativa, non dimostrano causalità; ST è PAVE-inspired, non una riproduzione completa di PAVE.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Esperimento 1 — A1-R8 contro A1_VERIFY

Risultati di **8 casi development appaiati**, 16 run, commit sperimentale `6927fcfa18fc29fce64ad65eac98231b2ba42184`. La sola analisi seguente è offline, dallo scoring ufficiale preesistente. Verranno mostrati score per incidente, token e secondi.

In [ ]:
from pathlib import Path
import hashlib, json, zipfile
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

root = Path("/content/drive/MyDrive/Tesi_OpenRCA")
FIGURES = Path("/content/relatore_grafici_2026_11_03")
FIGURES.mkdir(exist_ok=True, parents=True)

def verified_json(relative, sha):
    path = root / relative
    raw = path.read_bytes()
    assert hashlib.sha256(raw).hexdigest() == sha, "SHA-256 differente: " + str(path)
    return json.loads(raw.decode("utf-8"))

def save_plot(fig, name):
    path = FIGURES / (name + ".png")
    fig.savefig(path, dpi=170, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("PNG:", path)

freeze1 = verified_json(
    "data/protocols/a1_verify_paired_development_v1_results_freeze_v1.json",
    "e8c350d3e3372d0e4afb6fd63d60f2f5632394bc7e2a4b996ca0ff864ae2ccf0",
)
scoring1 = verified_json(
    "data/results/scoring/a1_verify_paired_development_v1_official_scoring_v1.json",
    "2504295d89f49c20c5e14da552ee176e54e52286cc2f86c417755367a814fb5f",
)
assert freeze1["git_commit"] == "6927fcfa18fc29fce64ad65eac98231b2ba42184"
assert scoring1["results_freeze_sha256"] == "e8c350d3e3372d0e4afb6fd63d60f2f5632394bc7e2a4b996ca0ff864ae2ccf0"
r1 = pd.DataFrame(scoring1["rows"])
assert len(r1) == 16 and set(r1["variant"]) == {"A1","A1_VERIFY"}
assert not r1.duplicated(["case_id","variant"]).any()
p1 = r1.pivot(index="case_id",columns="variant",values="official_score")[["A1","A1_VERIFY"]].sort_index()
assert p1.shape == (8,2) and p1.notna().all().all()
p1["delta_VERIFY"] = p1["A1_VERIFY"] - p1["A1"]
s1 = scoring1["summary"]
assert abs(s1["A1"]["mean_score"] - 0.3850) < 0.0002
assert abs(s1["A1_VERIFY"]["mean_score"] - 0.3225) < 0.0002
assert (s1["A1"]["total_tokens"],s1["A1_VERIFY"]["total_tokens"]) == (474590,258649)
display(p1.round(3))
print("Score medio:",s1["A1"]["mean_score"],s1["A1_VERIFY"]["mean_score"])
print("JSON validi:",s1["A1"]["json_valid"],"/8 vs",s1["A1_VERIFY"]["json_valid"],"/8")
print("Freeze e scoring primo studio: SHA-256 corretti.")

In [ ]:
fig,ax=plt.subplots(figsize=(11,4.3))
p1[["A1","A1_VERIFY"]].plot.bar(ax=ax)
ax.set(title="Esperimento 1 — score ufficiale per caso",xlabel="Incidente development",ylabel="Score OpenRCA")
ax.legend(["A1-R8","A1_VERIFY"])
ax.set_ylim(0,1.08)
ax.tick_params(axis="x",rotation=40)
save_plot(fig,"01_score_casi_r8_verify")

def paired_bars(title,variants,vals,unit,filename):
    fig,ax=plt.subplots(figsize=(7,4))
    bars=ax.bar(variants,vals)
    ax.set(title=title,ylabel=unit)
    ax.set_ylim(0,max(vals)*1.22 if max(vals)>0 else 1)
    for b,v in zip(bars,vals):
        ax.annotate(f"{v:,.1f}" if isinstance(v,float) else f"{v:,}",
                    (b.get_x()+b.get_width()/2,b.get_height()),
                    xytext=(0,4),textcoords="offset points",ha="center")
    save_plot(fig,filename)

paired_bars("Esperimento 1 — token totali",["A1-R8","A1_VERIFY"],
            [s1["A1"]["total_tokens"],s1["A1_VERIFY"]["total_tokens"]],
            "Token","02_token_r8_verify")
paired_bars("Esperimento 1 — tempo end-to-end totale",["A1-R8","A1_VERIFY"],
            [s1["A1"]["end_to_end_seconds"],s1["A1_VERIFY"]["end_to_end_seconds"]],
            "Secondi","03_tempo_r8_verify")
print("Riduzione token:",f"{100*(1-s1['A1_VERIFY']['total_tokens']/s1['A1']['total_tokens']):.2f}%")
print("Riduzione tempo:",f"{100*(1-s1['A1_VERIFY']['end_to_end_seconds']/s1['A1']['end_to_end_seconds']):.2f}%")
print("Interpretazione: più efficienza, NON maggiore score medio.")

## 3. Esperimento 2 — A1_VERIFY contro A1_VERIFY_ST

**Altri otto casi development appaiati, 16 run**, commit sperimentale `138fb523efec4a665a2424ab2faedb698f10562e`. Entrambi: stessi otto detector, cinque tool, modello e budget 4+2; la sola differenza progettuale è il contesto strutturale-temporale aggiunto alla finalizzazione.

Le tre celle seguenti sono riprese dalla demo ST verificata il 2 novembre. Accertano hash dei manifest, integrità delle 16 predizioni, parità di evidence SHA entro ciascuna coppia e scoring già archiviato.

In [ ]:
from pathlib import Path
from collections import defaultdict
import hashlib, json
import pandas as pd

root = Path("/content/drive/MyDrive/Tesi_OpenRCA")
protocol = "a1_verify_st_paired_development_v1"
freeze_path = root / "data/protocols" / f"{protocol}_predictions_frozen_v1.json"
score_path = root / "data/results" / f"{protocol}_official_scoring_v1.json"
EXPECTED_FREEZE = "43e646792685b1cde03052cadc2c8a82749d058a4e04ee0b01f607a7384fd104"
EXPECTED_SCORING = "26f10169e379f9a8ceabcd83923ae7daa72ca90f705e021d77e6eb5772027f45"
EXPECTED_COMMIT = "138fb523efec4a665a2424ab2faedb698f10562e"

def sha256(raw):
    return hashlib.sha256(raw).hexdigest()

assert sha256(freeze_path.read_bytes()) == EXPECTED_FREEZE
assert sha256(score_path.read_bytes()) == EXPECTED_SCORING
freeze = json.loads(freeze_path.read_text(encoding="utf-8"))
scoring = json.loads(score_path.read_text(encoding="utf-8"))
assert freeze["protocol"] == protocol == scoring["protocol"]
assert freeze["git_commit"] == EXPECTED_COMMIT
assert freeze["run_count"] == 16 and len(freeze["results"]) == 16
assert len(scoring["results"]) == 16
print("Freeze e scoring verificati con SHA-256. Nessuna API chiamata.")

In [ ]:
records = []
pair_sha = defaultdict(list)
seen = set()
for item in freeze["results"]:
    run_id = item["run_id"]
    assert run_id not in seen
    seen.add(run_id)
    path = root / item["result_relative_path"]
    raw = path.read_bytes()
    assert sha256(raw) == item["result_file_sha256"]
    result = json.loads(raw)
    provenance = result["paired_provenance"]
    assert provenance["run_id"] == run_id
    assert provenance["git_commit"] == EXPECTED_COMMIT
    assert provenance["integrity_issues"] == []
    assert result["error"] is None
    assert sha256(result["raw_response"].encode("utf-8")) == item["raw_response_sha256"]
    assert result["variant"] == item["variant"]
    assert result["case_id"] == item["case_id"]
    m = result["metrics"]
    assert m["token_usage_complete"] is True
    assert 0 <= m["exploration_tool_rounds"] <= 4
    assert 0 <= m["verification_tool_rounds"] <= 2
    pair_sha[item["case_id"]].append(result["evidence_sha256"])
    if result["variant"] == "A1_VERIFY_ST":
        assert isinstance(result.get("structural_temporal_context"), dict)
    else:
        assert result.get("structural_temporal_context") is None
    records.append({
        "case_id": item["case_id"],
        "variant": item["variant"],
        "json_status": result["json_status"],
        "tokens": m["total_tokens"],
        "seconds": m["end_to_end_seconds"],
        "checks": m["verification_tool_rounds"],
    })
assert len(pair_sha) == 8 and all(len(v) == 2 and v[0] == v[1] for v in pair_sha.values())
runs = pd.DataFrame(records)
display(runs.sort_values(["case_id", "variant"]).reset_index(drop=True))
print("Audit completato: 8 coppie, 16 risultati e stessa evidenza iniziale per coppia.")

In [ ]:
scores = pd.DataFrame(scoring["results"])
assert set(scores["variant"]) == {"A1_VERIFY", "A1_VERIFY_ST"}
assert not scores.duplicated(["case_id","variant"]).any()
pivot = scores.pivot(index="case_id", columns="variant", values="score")
assert pivot.shape == (8, 2)
pivot["delta_ST"] = pivot["A1_VERIFY_ST"] - pivot["A1_VERIFY"]
display(pivot.round(2))
wins = int((pivot["delta_ST"] > 1e-9).sum())
losses = int((pivot["delta_ST"] < -1e-9).sum())
ties = int((pivot["delta_ST"].abs() <= 1e-9).sum())
print(f"ST: {wins} caso migliorato, {losses} peggiorato, {ties} pari")
print("Score medio VERIFY:", round(pivot["A1_VERIFY"].mean(), 4))
print("Score medio ST:", round(pivot["A1_VERIFY_ST"].mean(), 4))
print("Differenza media ST−VERIFY:", round(pivot["delta_ST"].mean(), 4))
print("Output legacy validi:", runs.assign(valid=runs["json_status"]=="valid_json").groupby("variant")["valid"].sum().to_dict())
print("Token totali:", runs.groupby("variant")["tokens"].sum().to_dict())
print("Secondi totali:", runs.groupby("variant")["seconds"].sum().round(3).to_dict())

In [ ]:
fig,ax=plt.subplots(figsize=(11,4.3))
pivot["delta_ST"].plot.bar(ax=ax)
ax.axhline(0,color="black",linewidth=0.8)
ax.set(title="Esperimento 2 — delta score per caso (ST − VERIFY)",
       xlabel="Incidente development",ylabel="Delta score OpenRCA")
ax.tick_params(axis="x",rotation=40)
save_plot(fig,"04_delta_score_st")
stat = runs.groupby("variant")[["tokens","seconds"]].sum()
assert (int(stat.loc["A1_VERIFY","tokens"]),int(stat.loc["A1_VERIFY_ST","tokens"])) == (318154,349266)
for key,title,unit,suffix in [
    ("tokens","Esperimento 2 — token totali","Token","05_token_st"),
    ("seconds","Esperimento 2 — tempo end-to-end totale","Secondi","06_tempo_st")
]:
    vals=[stat.loc[v,key] for v in ("A1_VERIFY","A1_VERIFY_ST")]
    paired_bars(title,["A1_VERIFY","A1_VERIFY_ST"],vals,unit,suffix)
print("Delta token ST:",f"{100*(stat.loc['A1_VERIFY_ST','tokens']/stat.loc['A1_VERIFY','tokens']-1):+.2f}%")
print("Delta tempo ST:",f"{100*(stat.loc['A1_VERIFY_ST','seconds']/stat.loc['A1_VERIFY','seconds']-1):+.2f}%")
print("Attenzione: 1 migliorato, 1 peggiorato, 6 invariati; l'effetto non è robusto.")
print("NON confrontare lo score medio del 1° studio con quello del 2°.")

## 4. Perché R03, R04 e R05 non sono state implementate

OpenRCA Market contiene **15 classi di guasto**: nel nostro protocollo abbiamo 8 regole attive (R01/R02/R06–R11), tre categorie di rete **non ancora validate** (R03: ritrasmissione pacchetti container; R04: corruzione pacchetti container; R05: latenza di rete container) e quattro categorie tenute OOD e riservate.

Le fasi A/1–A/6 hanno ispezionato **quattro finestre development già viste** (due per cloudbed), senza ground truth né Gemini. Sono controlli di **osservabilità**, non test di accuratezza o causalità.

I conteggi sottostanti sono quelli comunicati dall'output Colab del **3 novembre 2026**. Se i JSON integrali A/4 e A/6 sono reperibili in `/content` o in `Tesi_OpenRCA/demo_relatore/audit_2026_11_03`, la prossima cella verifica che corrispondano; altrimenti i grafici rimangono esplicitamente basati sullo snapshot trascritto.

In [ ]:
from collections import Counter

# Snapshot dei conteggi A/4 e A/6 riferiti durante la verifica su Colab.
# NON sono score OpenRCA e non sono nuove diagnosi.
r03 = {
    "Entrambi positivi":274,
    "Solo retrans_segs":116,
    "Solo retrans_pct":146,
    "Entrambi non positivi":184,
}
r05 = {
    "Entrambi positivi":14679,
    "Solo durata":4,
    "Solo richieste":2558,
    "Entrambi non positivi":16719,
}
r04 = {
    "Errori RX/TX container":(336,336),
    "Packet drop container":(336,336),
    "Errori packet nodo":(48,48),
}
assert sum(r03.values()) == 720
assert sum(r05.values()) == 33960
assert sum(x[0] for x in r04.values()) == 720

search_dirs=[Path("/content"),root/"demo_relatore/audit_2026_11_03"]
def optional_artifact(name):
    for directory in search_dirs:
        path=directory/name
        if path.is_file(): return path
    return None

a6_file=optional_artifact("network_temporal_alignment_v1.json")
if a6_file is not None:
    a6=json.loads(a6_file.read_text(encoding="utf-8"))
    assert a6["audit"]=="network_temporal_alignment_v1"
    assert a6["new_case_count"]==0 and a6["read_ground_truth"] is False
    assert a6["gemini_calls"]==0 and a6["classification_performed"] is False
    x,y=a6["r03"]["total"],a6["r05"]["total"]
    assert [x["both_positive_same_timestamp"],x["first_positive_second_nonpositive"],
            x["second_positive_first_nonpositive"],x["neither_positive"]]==list(r03.values())
    assert [y["both_positive_same_timestamp"],y["first_positive_second_nonpositive"],
            y["second_positive_first_nonpositive"],y["neither_positive"]]==list(r05.values())
    assert x["both_numeric_pairs"]==720 and y["both_numeric_pairs"]==33960
    print("A/6 verificato dal file integrale:",a6_file)
else:
    print("NOTA: A/6 usa conteggi dell'output Colab comunicato il 03/11; JSON non disponibile.")
a4_file=optional_artifact("network_feasibility_summary_v1.json")
if a4_file is not None:
    a4=json.loads(a4_file.read_text(encoding="utf-8"))
    assert a4["audit"]=="network_feasibility_summary_v1"
    assert a4["read_ground_truth"] is False and a4["read_market_zip"] is False
    for family,label in [
        ("container_rx_tx_errors","Errori RX/TX container"),
        ("container_dropped_control","Packet drop container"),
        ("node_packet_errors","Errori packet nodo"),
    ]:
        subset=[z for z in a4["group_summary"] if z["family"]==family]
        assert sum(z["profiles"] for z in subset)==r04[label][0]
        assert sum(z["all_zero_profiles"] for z in subset)==r04[label][1]
        assert sum(z["positive_max_profiles"] for z in subset)==0
    print("A/4 verificato dal file integrale:",a4_file)
else:
    print("NOTA: A/4 usa conteggi dell'output Colab comunicato il 03/11; JSON non disponibile.")

In [ ]:
def distribution_chart(values,title,ylabel,filename):
    fig,ax=plt.subplots(figsize=(10,4))
    bars=ax.bar(list(values),list(values.values()))
    ax.set(title=title,ylabel=ylabel)
    ax.set_ylim(0,max(values.values())*1.18)
    ax.tick_params(axis="x",rotation=17)
    for bar,n in zip(bars,values.values()):
        ax.annotate(f"{n:,}".replace(",","."),(bar.get_x()+bar.get_width()/2,bar.get_height()),
                    xytext=(0,4),textcoords="offset points",ha="center")
    save_plot(fig,filename)

distribution_chart(r03,"R03 — ritrasmissioni TCP: coppie allo stesso timestamp",
                   "Coppie nodo-finestra-timestamp (720)","07_r03_tcp_node")
print("R03: i KPI sono a livello NODE e non identificano il container root cause.")

families=list(r04)
total=[r04[x][0] for x in families]
positive=[r04[x][0]-r04[x][1] for x in families]
fig,ax=plt.subplots(figsize=(10,4))
bars=ax.bar(families,total,label="Profili con massimo zero")
ax.bar(families,positive,label="Profili con massimo positivo")
for bar,n in zip(bars,total):
    ax.annotate(f"{n} tutti zero",(bar.get_x()+bar.get_width()/2,bar.get_height()),
                xytext=(0,4),textcoords="offset points",ha="center")
ax.set(title="R04 — errori e drop: 720/720 profili sempre nulli",
       ylabel="Profili per famiglia (non singoli campioni)")
ax.set_ylim(0,max(total)*1.22)
ax.tick_params(axis="x",rotation=12)
ax.legend()
save_plot(fig,"08_r04_segnali_zero")
print("R04: nessuna evidenza discriminante nel campione; non prova assenza in tutto Market.")

distribution_chart(r05,"R05 — Istio: durata RPC e conteggio richieste",
                   "Coppie componente-suffisso-timestamp (33.960)","09_r05_istio")
print("R05: durata di richiesta NON equivale alla sola latenza di rete.")

## 5. Conclusioni da discutere con il professore

| Regola | Evidenza osservata A/6 | Perché esclusa per ora |
|---|---|---|
| **R03** | **720/720** coppie TCP complete; **274** entrambe positive | Il segnale è **node-level**, mentre la classe richiesta è **container-level**; unità e semantica non ancora dimostrate |
| **R04** | A/4: **720/720** profili di errori/drop sempre zero | I KPI sono costanti nel campione; errori generici non dimostrano *packet corruption* |
| **R05** | **33.960/33.960** coppie Istio complete; **14.679** entrambe positive | La durata RPC include eventualmente calcolo lato server; non è misura isolata di **network latency** |

**Proposta:** restare sulle **8 regole validate come implementazione software** (non su 11 detector di guasto già dimostrati). R03 può fungere da indizio ausiliario node-level; R04 resta sospesa; R05 è candidata a uno studio separato su trace parent-child.

Gli 8 detector esistenti hanno test software, ma **non è stata dimostrata un'accuratezza di classe indipendente**. Il *knowledge transfer* dell'harness è una **ipotesi futura**, non un risultato già misurato. Gli esperimenti sono development, non OOD blind.

**Domande per il relatore:** concentrarci sul contributo misurato dell'harness? Riportare il *negative result* R04? Serve davvero un nuovo test trace prima della scrittura della tesi entro il **1 dicembre 2026**?

In [ ]:
# I PNG vengono scritti solo nella cartella temporanea /content, non nei freeze.
out=Path("/content/relatore_grafici_2026_11_03.zip")
with zipfile.ZipFile(out,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for png in sorted(FIGURES.glob("*.png")):
        z.write(png,arcname=png.name)
print(f"Generati {len(list(FIGURES.glob('*.png')))} grafici PNG.")
print("Pacchetto per la presentazione:",out)
print("Nessuna API, scorer, ground truth o modifica dei risultati frozen.")